In [1]:
############### Plot time trend of environmental variables #########

import pandas as pd
import matplotlib.pyplot as plt
from itertools import cycle
from matplotlib import rcParams
from scipy.stats import linregress
import requests
import numpy as np
from numpy import triu_indices_from, std

In [2]:
def get_vertcon_offset(lat, lon, units="meters"):
    url = "https://geodesy.noaa.gov/api/ncat/transform"
    params = {
        "from_h": "NGVD29",
        "to_h": "NAVD88",
        "from_lat": lat,
        "from_lon": lon,
        "from_z": 0,
        "to_z": 0,
        "height_units": units,
        "output_units": units
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["dh"]  # Offset in same units

In [ ]:
# Set global font to Times New Roman
%matplotlib agg
rcParams['font.family'] = 'Times New Roman'
output_path = '/Users/bz294/Documents/WhiteZone/Figure/Annual_avg_salinity.tiff'
fs = 16
# Read the CSV file
df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/salinity_annual_avg.csv')

# Clean site names
df['site'] = df['site'].str.replace('_salinity$', '', regex=True)

# Remove sites starting with 'S197'
df = df[~df['site'].str.match("^S197")]

# Convert types
df['year'] = df['year'].astype(int)
df = df[df['year'] != 2025]
df['annual_avg'] = pd.to_numeric(df['annual_avg'], errors='coerce')

# Define color palette
custom_colors = ['#d53e4f', '#fc8d59', '#fee08b', '#ffffbf', '#e6f598', '#99d594', '#3288bd']
color_cycle = cycle(custom_colors)

# Plotting
fig, ax = plt.subplots(figsize=(10, 6))

for site in df['site'].unique():
    site_data = df[df['site'] == site]
    ax.plot(site_data['year'], site_data['annual_avg'], label=site, color=next(color_cycle))

ax.set_xlabel('Year', fontsize=fs)
ax.set_ylabel('Annual Average Salinity (PSU)', fontsize=fs)
ax.set_title('Annual Average Salinity Time Series by Site', fontsize=fs)
ax.tick_params(axis='both', labelsize=fs)  # change 12 to your preferred size

# Move legend to the left, outside the plot
ax.legend(bbox_to_anchor=(1.15, 1), loc='upper left', borderaxespad=0, fontsize = fs-2)

ax.grid(True)
plt.tight_layout()
plt.show()
plt.savefig(output_path, dpi=300)


In [ ]:
#### Calculate pairwise coefficient ###

# Pivot so each column = one site, rows = years
pivot_df = df.pivot_table(index='year', columns='site', values='annual_avg')

# Keep only years with complete data across all sites
common_years_df = pivot_df.dropna()

# Compute Pearson correlation matrix
corr_matrix = common_years_df.corr(method='pearson')

# Extract upper triangle (excluding diagonal)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Compute average pairwise correlation
avg_corr = upper_tri.stack().mean()

print("Average pairwise Pearson correlation:", avg_corr)


In [ ]:
for site in df['site'].unique():
    site_data = df[df['site'] == site].dropna(subset=['annual_avg', 'year'])
    x = site_data['year']
    y = site_data['annual_avg']
    
    if len(x) < 2:
        print(f"Site {site}: Not enough data points to fit linear regression.")
        continue
    
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    
    print(f"Site {site}: y = {slope:.4f} * x + {intercept:.4f} (R² = {r_value**2:.3f})")

In [ ]:
############### Plot the EDEN sites water level together ############
%matplotlib agg
output_path = '/Users/bz294/Documents/WhiteZone/Figure/Annual_avg_EDEN_wl.tiff'
# Read the annual summary CSV for water levels
# Read the CSV
plt.clf()
hydro_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/EDEN/annual_summary.csv')

# Keep only specified sites
keep_sites = ['CT27R', 'CT50R', 'EPSW', 'EVER6', 'EVER7', 'EVER8', 'G-1251', 'Upstream_Taylor']
hydro_df = hydro_df[hydro_df['source_file'].isin(keep_sites)]


# Convert types
hydro_df['year'] = hydro_df['year'].astype(int)
hydro_df = hydro_df[hydro_df['year'] != 2025]
hydro_df['water_level'] = pd.to_numeric(hydro_df['mean_value'], errors='coerce') * 30.48

# Colors
custom_colors = ['#d53e4f', '#fc8d59', '#fee08b', '#ffffbf', '#e6f598', '#99d594', '#3288bd']
color_cycle = cycle(custom_colors)

# Plotting
fig, ax = plt.subplots(figsize=(10, 6))

for source in keep_sites:
    source_data = hydro_df[hydro_df['source_file'] == source].dropna(subset=['year', 'water_level'])
    ax.plot(source_data['year'], source_data['water_level'], label=source, color=next(color_cycle))

ax.set_xlabel('Year', fontsize=fs)
ax.set_ylabel('Water Level (cm)', fontsize=fs)
ax.set_title('EDEN Water Level Time Series by Sites', fontsize=fs)
ax.tick_params(axis='both', labelsize=fs)

# Legend outside on the left
ax.legend(bbox_to_anchor=(-0.15, 1), loc='upper right', borderaxespad=0, fontsize = fs-2)

ax.grid(True)
plt.tight_layout()
plt.show()
plt.savefig(output_path, dpi=300)

In [ ]:
################# Plot DBHydro water level after converting datum #########
# Read the annual summary (hydrology time series)
dbhydro_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/DBhydro/annual_summary.csv')

# Read the site metadata with datum info
datum_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Hydro/DBhydro/WZ_hydro_sites_latlon.csv')

dbhydro_df = dbhydro_df.merge(
    datum_df[['Name', 'Datum', 'Conversion']],
    left_on='source_file',
    right_on='Name',
    how='left'
)

# Drop the redundant 'Name' column from the merge
dbhydro_df.drop(columns=['Name'], inplace=True)

# Preview result
print(dbhydro_df.head())

# Convert mean_value to NAVD88 only when Datum is '29'
dbhydro_df['mean_value_NAVD88'] = dbhydro_df.apply(
    lambda row: row['mean_value'] + row['Conversion'] if str(row['Datum']).strip() == '29' else row['mean_value'],
    axis=1
)

# Preview the result

dbhydro_df['mean_value_NAVD88'] = dbhydro_df['mean_value_NAVD88'] * 30.48
print(dbhydro_df[['source_file', 'year', 'Datum', 'Conversion', 'mean_value', 'mean_value_NAVD88']].head())

In [ ]:
%matplotlib inline

plt.clf()
# Drop any rows with missing values in required columns
plot_df = dbhydro_df.dropna(subset=['source_file', 'year', 'mean_value_NAVD88'])

# Get unique site names
sites = plot_df['source_file'].unique()

# Set custom color cycle
custom_colors = ['#d53e4f', '#fc8d59', '#fee08b', '#ffffbf', '#e6f598', '#99d594', '#3288bd']
color_cycle = cycle(custom_colors)

# Set up figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot each site's time series
for site in sites:
    site_data = plot_df[plot_df['source_file'] == site]
    ax.plot(site_data['year'], site_data['mean_value_NAVD88'], label=site, color=next(color_cycle))

# Labeling
ax.set_xlabel('Year', fontsize=fs, fontname='Times New Roman')
ax.set_ylabel('Water Level (cm)', fontsize=fs, fontname='Times New Roman')
ax.set_title('DBHydro Water Level Time Series by Site ', fontsize=fs, fontname='Times New Roman')
ax.tick_params(axis='both', labelsize=fs)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontname('Times New Roman')

# Legend outside to the left
ax.legend(title='Site', bbox_to_anchor=(-0.15, 1), loc='upper right', borderaxespad=0, prop={'family': 'Times New Roman', 'size': 10})

# Grid and layout
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
%matplotlib agg
plt.clf()

hydro_df = hydro_df.rename(columns={"water_level": "mean_value_NAVD88"})
output_path = '/Users/bz294/Documents/WhiteZone/Figure/Annual_avg_wl_combined_.tiff'
# Select only the needed columns from each DataFrame
df1 = dbhydro_df[['year', 'mean_value_NAVD88', 'source_file']].copy()
df2 = hydro_df[['year', 'mean_value_NAVD88', 'source_file']].copy()

# Combine them
combined_df = pd.concat([df1, df2], ignore_index=True)

combined_df = combined_df[(combined_df['year'] >= 1984) & (combined_df['year'] < 2025)]

# Drop rows with missing values in critical columns
combined_df = combined_df.dropna(subset=['year', 'mean_value_NAVD88', 'source_file'])

# Get unique sites
sites = combined_df['source_file'].unique()

# Setup colors
custom_colors = ['#a6cee3','#1f78b4','#b2df8a','#33a02c','#fb9a99','#e31a1c','#fdbf6f','#ff7f00','#cab2d6','#6a3d9a','#ffff99','#b15928']
sites = sorted(combined_df['source_file'].unique())

# Setup colors
color_cycle = cycle(custom_colors)
site_colors = {site: next(color_cycle) for site in sites}

# Plot
plt.figure(figsize=(12, 6))
for site in sites:
    site_data = combined_df[combined_df['source_file'] == site]
    plt.plot(site_data['year'], site_data['mean_value_NAVD88'], label=site, color=site_colors[site])


plt.xlabel('Year', fontsize=fs, fontname='Times New Roman')
plt.ylabel('Water Level (cm)', fontsize=fs, fontname='Times New Roman')
plt.title('Annual Average Water Level Time Series by Site', fontsize=fs, fontname='Times New Roman')
plt.xticks(fontsize=fs, fontname='Times New Roman')
plt.yticks(fontsize=fs, fontname='Times New Roman')

ax = plt.gca()  # get current axes

xticks = np.arange(1984, 2025, step=5)  # ticks every 5 years from 1985 to 2025
ax.set_xticks(xticks)
ax.set_xlim(1984, 2024)

plt.legend(
    bbox_to_anchor=(1.15, 1),
    loc='upper left',
    borderaxespad=0,
    prop={'family': 'Times New Roman', 'size': fs - 6},
    ncol=2
)

plt.grid(True)
plt.tight_layout()
plt.show()
plt.savefig(output_path, dpi=300)

In [ ]:
# --- NEW CODE: Calculate Correlation for Common Time Period ---

# 1. Pivot the data to a wide format (one column per gauge)
pivot_df = combined_df.pivot(index='year', columns='source_file', values='mean_value_NAVD88')

# 2. Find the common time period (where NO gauge has NaN)
# Drop years where ANY gauge is missing data
common_time_df = pivot_df.dropna(how='any')

if common_time_df.empty:
    print("Warning: There is no single year where all gauges have data. Cannot calculate correlation for common period.")
else:
    # 3. Calculate the correlation matrix on the common-time DataFrame
    correlation_matrix = common_time_df.corr(method='pearson')
    
    # 4. Print the results
    print("Common Time Period:", common_time_df.index.min(), "to", common_time_df.index.max())
    print("\nPearson Correlation Coefficient Matrix (Common Time Period):")
    
# Assuming 'correlation_matrix' is your DataFrame of correlations

# 1. Calculate the Average Pairwise Correlation
# Get the upper triangle of the matrix (excluding the diagonal of 1s)
corr_values = correlation_matrix.values
upper_triangular_indices = triu_indices_from(corr_values, k=1)
average_correlation = corr_values[upper_triangular_indices].mean()

print(average_correlation)

In [ ]:
num_sites = combined_df['source_file'].nunique()
print(f"Number of unique sites: {num_sites}")

In [ ]:
############ Output annual average #####

# Filter for years between 1984 and 2024 (inclusive)
filtered_df = combined_df[(combined_df["year"] >= 1984) & (combined_df["year"] <= 2024)]

# Group by 'year' and calculate the mean of 'mean_value_NAVD88'
annual_navd88_mean = filtered_df.groupby("year")["mean_value_NAVD88"].mean().reset_index()

# Rename column for clarity
annual_navd88_mean.columns = ["year", "annual_mean_NAVD88"]

# Save to CSV
output_path = "annual_mean_NAVD88_1984_2024.csv"
annual_navd88_mean.to_csv(output_path, index=False)

# Optional: preview the result
print(annual_navd88_mean)

In [ ]:
########### Temporal and spatial analysis based on the WL gauge measurements #####
combined_df
filtered_df = combined_df[
    (combined_df['source_file'].isin(['NP-146', 'NP-EPSW'])) &
    (combined_df['year'] > 2013)
]
np146_values = filtered_df[filtered_df['source_file'] == 'NP-146']['mean_value_NAVD88'].dropna()
npepsw_values = filtered_df[filtered_df['source_file'] == 'NP-EPSW']['mean_value_NAVD88'].dropna()

In [ ]:
np146_values

In [ ]:
npepsw_values